In [1]:
import torch
from torch import nn
from torch.nn import functional as F
import numpy as np
import pandas as pd
import os
import PIL
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import tarfile
from pathlib import Path
from torchvision.transforms import transforms
from torchvision.datasets import ImageFolder
from d2l import torch as d2l
from torchvision.models import resnet18

In [2]:
data_path=Path("D:/Pytorch/data/flower_photos/raw/data/")
data_path.mkdir(exist_ok=True)

In [3]:
train_dataset=ImageFolder(
    root="D:/Pytorch/data/flower_photos/raw/data/flower_photos/flower_photos/train",
    transform=transforms.Compose([
        transforms.Resize((96,96)),
        transforms.RandomHorizontalFlip(0.5),
        transforms.RandomResizedCrop((96,96),scale=(0.8,1.0)),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor()
    ])
)
test_dataset=ImageFolder(
    root="D:/Pytorch/data/flower_photos/raw/data/flower_photos/flower_photos/test",
    transform=transforms.Compose([
        transforms.Resize((96,96)),
        transforms.ToTensor()
    ])
)
validation_dataset=ImageFolder(
    root="D:/Pytorch/data/flower_photos/raw/data/flower_photos/flower_photos/validation",
    transform=transforms.Compose([
        transforms.Resize((96,96)),
        transforms.ToTensor()
    ])
)
train_len=len(train_dataset)
test_len=len(test_dataset)
validation_len=len(validation_dataset)
print(f"训练集大小为{train_len}")
print(f"测试集大小为{test_len}")
print(f"验证集大小为{validation_len}")

训练集大小为3540
测试集大小为50
验证集大小为80


In [4]:
img_shape=train_dataset[0]
print(f"图像的形状为{img_shape[0].shape}")
print(f"类别数量为{len(train_dataset.classes)}")
print(f"类别名称为{train_dataset.classes}")

图像的形状为torch.Size([3, 96, 96])
类别数量为5
类别名称为['daisy', 'dandelion', 'roses', 'sunflowers', 'tulips']


In [5]:
DEVICE=('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE=64
lr=0.001

In [6]:
train_load=torch.utils.data.DataLoader(train_dataset,batch_size=BATCH_SIZE,shuffle=True)
test_load=torch.utils.data.DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False)
validation_load=torch.utils.data.DataLoader(validation_dataset,batch_size=BATCH_SIZE,shuffle=False)

In [7]:

# net = nn.Sequential(
#     nn.Conv2d(3, 32, kernel_size=3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
#     nn.MaxPool2d(2, 2),
#     nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
#     nn.MaxPool2d(2, 2),
#     nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
#     nn.MaxPool2d(2, 2),
#     nn.Flatten(),
#     nn.Dropout(0.5),
#     nn.Linear(128 * 12 * 12, 5)
# )

In [8]:
net=resnet18(pretrained=True)
net.fc=nn.Linear(net.fc.in_features,5)
net=net.to(DEVICE)

d:\Pytorch\.venv_cuda\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
d:\Pytorch\.venv_cuda\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [9]:
loss=nn.CrossEntropyLoss()
optimizer=torch.optim.AdamW(net.parameters(),lr=lr,weight_decay=1e-4)

In [10]:
def train(net,train_iter,test_iter,num_epochs):
    net.to(DEVICE)
    print("training on ",DEVICE)
    for epoch in range(num_epochs):
        net.train()
        metric=d2l.Accumulator(3)
        for (X,y) in train_iter:
            optimizer.zero_grad()
            X,y=X.to(DEVICE),y.to(DEVICE)
            y_hat=net(X)
            l=loss(y_hat,y)
            l.backward()
            optimizer.step()
            metric.add(l.item()*X.shape[0],d2l.accuracy(y_hat,y),X.shape[0])
        train_loss=metric[0]/metric[2]
        train_acc=metric[1]/metric[2]
        print(f"epoch{epoch+1},loss{train_loss:.3f},train_acc{train_acc:.3f}")

        net.eval()
        test_metric = d2l.Accumulator(2)
        with torch.no_grad():
            for X, y in test_iter:
                X, y = X.to(DEVICE), y.to(DEVICE)
                y_hat = net(X)
                test_metric.add(d2l.accuracy(y_hat, y), X.shape[0])
        test_acc = test_metric[0] / test_metric[1]
        print(f"测试集准确率: {test_acc:.3f}")

In [11]:
train(net,train_load,test_load,num_epochs=20)

training on  cuda
epoch1,loss0.821,train_acc0.716
测试集准确率: 0.660
epoch2,loss0.462,train_acc0.841
测试集准确率: 0.740
epoch3,loss0.390,train_acc0.857
测试集准确率: 0.660
epoch4,loss0.350,train_acc0.875
测试集准确率: 0.620
epoch5,loss0.303,train_acc0.887
测试集准确率: 0.760
epoch6,loss0.242,train_acc0.909
测试集准确率: 0.700
epoch7,loss0.263,train_acc0.905
测试集准确率: 0.720
epoch8,loss0.228,train_acc0.921
测试集准确率: 0.520
epoch9,loss0.290,train_acc0.898
测试集准确率: 0.740
epoch10,loss0.227,train_acc0.920
测试集准确率: 0.740
epoch11,loss0.170,train_acc0.941
测试集准确率: 0.660
epoch12,loss0.176,train_acc0.936
测试集准确率: 0.800
epoch13,loss0.147,train_acc0.944
测试集准确率: 0.680
epoch14,loss0.142,train_acc0.950
测试集准确率: 0.720
epoch15,loss0.136,train_acc0.955
测试集准确率: 0.720
epoch16,loss0.130,train_acc0.955
测试集准确率: 0.740
epoch17,loss0.130,train_acc0.954
测试集准确率: 0.720
epoch18,loss0.104,train_acc0.964
测试集准确率: 0.760
epoch19,loss0.107,train_acc0.965
测试集准确率: 0.780
epoch20,loss0.094,train_acc0.968
测试集准确率: 0.800


In [12]:
def evaluate(net,validation_iter):
    net.eval()
    correct=0
    total=0
    with torch.no_grad():
        for X,y in validation_iter:
            X,y=X.to(DEVICE),y.to(DEVICE)
            y_hat=net(X)
            correct+=(y_hat.argmax(dim=1)==y).sum().item()
            total+=y.shape[0]
        return correct/total
    
validation_acc=evaluate(net,validation_load)
print(f"validation_acc{validation_acc:.3f}")

validation_acc0.925
